In [ ]:
import requests
import time
import json
import datetime
from kafka import KafkaProducer

print("Łączę się z brokerem pod 'broker:29092'...")
producer = KafkaProducer(
    bootstrap_servers='broker:29092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: str(k).encode('utf-8') if k else None
)
print("Połączono pomyślnie!\n")

API_KEY = 'f37dad49-20aa-4409-8336-0d87d19be0fa'
KAFKA_TOPIC = 'ztm_pozycje_gps'
URL = "https://api.um.warszawa.pl/api/action/busestrams_get/"
RESOURCE_ID = "f2e5503e-927d-4ad3-9500-4ab9e55eba59"

print(f"Rozpoczynam wysyłanie danych ZTM do topicu: {KAFKA_TOPIC}...")

try:
    while True:
        for vehicle_type in [1, 2]:  # 1 = autobusy, 2 = tramwaje
            try:
                params = {
                    "resource_id": RESOURCE_ID,
                    "apikey": API_KEY,
                    "type": vehicle_type
                }
                response = requests.get(URL, params=params, timeout=10)
                
                if response.status_code == 200:
                    dane = response.json()
                    pojazdy = dane.get('result', [])
                    
                    if isinstance(pojazdy, list):
                        for pojazd in pojazdy:
                            pojazd['vehicle_type'] = 'bus' if vehicle_type == 1 else 'tram'
                            pojazd['ingestion_time'] = datetime.datetime.now().isoformat()
                            key = pojazd.get('VehicleNumber', '')
                            producer.send(KAFKA_TOPIC, key=key, value=pojazd)
                        
                        producer.flush()
                        typ_str = "autobusów" if vehicle_type == 1 else "tramwajów"
                        print(f"[{time.strftime('%H:%M:%S')}] Wysłano {len(pojazdy)} {typ_str}.")
                    else:
                        print(f"Błąd z API ZTM: {pojazdy}")
                else:
                    print(f"Błąd HTTP: {response.status_code}")
            
            except requests.exceptions.RequestException as e:
                print(f"[{time.strftime('%H:%M:%S')}] Błąd sieci: {e}")
            except Exception as e:
                print(f"[{time.strftime('%H:%M:%S')}] Błąd: {e}")
        
        time.sleep(15)

except KeyboardInterrupt:
    print("\nZatrzymano przez użytkownika.")
finally:
    producer.close()

Łączę się z brokerem pod 'broker:29092'...
Połączono pomyślnie!

Rozpoczynam wysyłanie danych ZTM do topicu: ztm_pozycje_gps...
[17:43:40] Wysłano 1098 autobusów.
[17:43:40] Wysłano 292 tramwajów.
[17:43:55] Wysłano 1097 autobusów.
[17:43:55] Wysłano 292 tramwajów.
[17:44:11] Wysłano 1099 autobusów.
[17:44:11] Wysłano 293 tramwajów.
[17:44:26] Wysłano 1094 autobusów.
[17:44:26] Wysłano 291 tramwajów.
[17:44:42] Wysłano 1097 autobusów.
[17:44:42] Wysłano 292 tramwajów.
[17:44:57] Wysłano 1092 autobusów.
[17:44:57] Wysłano 292 tramwajów.
[17:45:12] Wysłano 1093 autobusów.
[17:45:13] Wysłano 292 tramwajów.
[17:45:28] Wysłano 1094 autobusów.
[17:45:28] Wysłano 292 tramwajów.
[17:45:43] Wysłano 1094 autobusów.
[17:45:43] Wysłano 292 tramwajów.
[17:45:59] Wysłano 1096 autobusów.
[17:45:59] Wysłano 291 tramwajów.
[17:46:14] Wysłano 1094 autobusów.
[17:46:14] Wysłano 292 tramwajów.
[17:46:29] Wysłano 1094 autobusów.
[17:46:30] Wysłano 292 tramwajów.
[17:46:45] Wysłano 1093 autobusów.
[17:46:45